# CS-272 Artificial Intelligence: Lab 4
### Search Problem Modeling - Tree Search vs Graph Search

**Name:** Arqam Waheed
**Enrollment No.:** 553514
**Class:** BSDS 2K25
**Lab:** 04

## Activity 1: Model the Search Problem and State Space

- **State:** the robot's current grid position, a `(row, col)` tuple
- **Actions:** North, South, West, East, only the ones that stay inside the grid and don't hit a wall
- **Transition model:** `result(state, action)` adds the move's offset to the position
- **Start state:** reception at (0, 0)
- **Goal test:** position equals the AI Lab at (4, 4)
- **Step cost:** 1 per legal move

**Why only the position?** The walls, the grid size and the goal never change during the search, so they belong to the problem, not the state. The only thing that differs from one moment to the next is where the robot is.

In [1]:
MOVES = {
    "North": (-1, 0),
    "South": (1, 0),
    "West":  (0, -1),
    "East":  (0, 1)
}

class CampusDeliveryProblem:
    def __init__(self):
        self.rows = 5
        self.cols = 5
        self.initial_state = (0, 0)
        self.goal_state = (4, 4)
        self.walls = {
            (0,3), (1,1), (1,3),
            (2,1), (3,3), (4,0)
        }

    def in_bounds(self, state):
        row, col = state
        return 0 <= row < self.rows and 0 <= col < self.cols

    def actions(self, state):
        valid = []
        for action in MOVES:
            candidate = self.result(state, action)
            if self.in_bounds(candidate) and candidate not in self.walls:
                valid.append(action)
        return valid

    def result(self, state, action):
        row, col = state
        dr, dc = MOVES[action]
        return (row + dr, col + dc)

    def goal_test(self, state):
        return state == self.goal_state

    def step_cost(self, state, action, next_state):
        return 1

    def state_space_size(self):
        return self.rows * self.cols - len(self.walls)

### Grid and state-space size
25 total cells - 6 walls = **19** legal states.

In [2]:
def show_world(problem, robot_state=None):
    for r in range(problem.rows):
        line = ""
        for c in range(problem.cols):
            pos = (r, c)
            if pos in problem.walls:
                line += "# "
            elif robot_state == pos:
                line += "R "
            elif pos == problem.initial_state:
                line += "S "
            elif pos == problem.goal_state:
                line += "G "
            else:
                line += ". "
        print(line)

problem = CampusDeliveryProblem()
show_world(problem, problem.initial_state)
print("Actions from start:", problem.actions(problem.initial_state))
print("State-space size:", problem.state_space_size())

R . . # . 
. # . # . 
. # . . . 
. . . # . 
# . . . G 
Actions from start: ['South', 'East']
State-space size: 19


This size counts unique positions the robot can be in, not search-tree nodes, because the tree can contain the same position many times through different paths.

### Checkpoint 2: change one wall
**Prediction:** removing the wall at (1,1) leaves 5 walls, so 25 - 5 = **20** states.

In [3]:
test = CampusDeliveryProblem()
test.walls.remove((1, 1))
show_world(test)
print("State-space size:", test.state_space_size())

S . . # . 
. . . # . 
. # . . . 
. . . # . 
# . . . G 
State-space size: 20


## Activity 2: Implement and Trace TREE_SEARCH vs GRAPH_SEARCH

In [4]:
class SearchNode:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def solution(self):
        actions = []
        node = self
        while node.parent is not None:
            actions.append(node.action)
            node = node.parent
        actions.reverse()
        return actions

def child_node(problem, parent, action):
    next_state = problem.result(parent.state, action)
    new_cost = parent.path_cost + problem.step_cost(parent.state, action, next_state)
    return SearchNode(next_state, parent=parent, action=action, path_cost=new_cost)

### TREE_SEARCH
No explored set, every generated child goes into the frontier.

In [5]:
def tree_search(problem, max_expansions=40):
    frontier = [SearchNode(problem.initial_state)]
    expanded_order = []
    while frontier and len(expanded_order) < max_expansions:
        node = frontier.pop(0)
        expanded_order.append(node.state)
        if problem.goal_test(node.state):
            return node, expanded_order
        for action in problem.actions(node.state):
            child = child_node(problem, node, action)
            frontier.append(child)
    return None, expanded_order

### GRAPH_SEARCH
Same FIFO frontier, but a child is added only if its state is not explored and not already in the frontier.

In [6]:
def graph_search(problem, max_expansions=40):
    start = SearchNode(problem.initial_state)
    frontier = [start]
    frontier_states = {problem.initial_state}
    explored = set()
    expanded_order = []
    while frontier and len(expanded_order) < max_expansions:
        node = frontier.pop(0)
        frontier_states.remove(node.state)
        expanded_order.append(node.state)
        if problem.goal_test(node.state):
            return node, expanded_order
        explored.add(node.state)
        for action in problem.actions(node.state):
            child = child_node(problem, node, action)
            if child.state not in explored and child.state not in frontier_states:
                frontier.append(child)
                frontier_states.add(child.state)
    return None, expanded_order

### Comparison

In [7]:
def report_search(name, result_node, expanded_order):
    print("\n---", name, "---")
    print("Goal found:", result_node is not None)
    print("Expanded nodes:", len(expanded_order))
    print("Unique expanded states:", len(set(expanded_order)))
    print("Repeated expansions:", len(expanded_order) - len(set(expanded_order)))
    if result_node is not None:
        print("Solution actions:", result_node.solution())
        print("Path cost:", result_node.path_cost)

problem = CampusDeliveryProblem()
tree_result, tree_trace = tree_search(problem)
graph_result, graph_trace = graph_search(problem)
report_search("TREE SEARCH", tree_result, tree_trace)
report_search("GRAPH SEARCH", graph_result, graph_trace)


--- TREE SEARCH ---
Goal found: False
Expanded nodes: 40
Unique expanded states: 9
Repeated expansions: 31

--- GRAPH SEARCH ---
Goal found: True
Expanded nodes: 18
Unique expanded states: 18
Repeated expansions: 0
Solution actions: ['South', 'South', 'South', 'East', 'South', 'East', 'East', 'East']
Path cost: 8


### Tree search trace (first 20 expansions)

In [8]:
print("First 20 expanded states:")
for i, state in enumerate(tree_trace[:20], 1):
    print(i, state)
print("(0, 0) expanded", tree_trace.count((0, 0)), "times")

First 20 expanded states:
1 (0, 0)
2 (1, 0)
3 (0, 1)
4 (0, 0)
5 (2, 0)
6 (0, 0)
7 (0, 2)
8 (1, 0)
9 (0, 1)
10 (1, 0)
11 (3, 0)
12 (1, 0)
13 (0, 1)
14 (1, 2)
15 (0, 1)
16 (0, 0)
17 (2, 0)
18 (0, 0)
19 (0, 2)
20 (0, 0)
(0, 0) expanded 9 times


### Graph search trace (complete)

In [9]:
print("Full expansion order:")
for i, state in enumerate(graph_trace, 1):
    print(i, state)

Full expansion order:
1 (0, 0)
2 (1, 0)
3 (0, 1)
4 (2, 0)
5 (0, 2)
6 (3, 0)
7 (1, 2)
8 (3, 1)
9 (2, 2)
10 (4, 1)
11 (3, 2)
12 (2, 3)
13 (4, 2)
14 (2, 4)
15 (4, 3)
16 (1, 4)
17 (3, 4)
18 (4, 4)


## Activity 3: Search Analysis Report

**1. Problem model:** `(row, col)` is enough because the goal and walls are fixed. Nothing else in the world changes, so two situations with the robot on the same cell are the same situation.

**2. State-space size:** 5 x 5 = 25 cells, 6 of them are walls, 25 - 6 = 19 legal states.

**3. Tree search:** (0, 0) was expanded 9 times in 40 expansions. It is expanded 1st as the start, then again 4th and 6th: South takes the robot to (1, 0) and North brings it straight back, and East to (0, 1) then West does the same. Out of 40 expansions only 9 were unique states, 31 were repeats, and the goal was not found.

**4. Graph search:** when a node is expanded its state goes into `explored`, so it is never added again. `frontier_states` holds the states still waiting, so a state reached by a second path is not queued twice. In my run this gave 18 expansions, 18 unique states and 0 repeats.

**5. Trade-off:** tree search stores only the frontier, graph search also stores `explored` and `frontier_states`. That memory is at most 19 states here, and it stops the search from re-expanding the same cells. Without it, reversible moves like South then North form cycles and the tree keeps growing forever. Graph search reached the goal in 18 expansions while tree search used all 40 and was still stuck near the start.

**6. Conclusion:** graph search was more reliable on this finite grid. It found the goal with path cost 8 (South, South, South, East, South, East, East, East) and never repeated a state, while tree search ran out of its expansion limit.

**State-space graph vs search tree:** the state-space graph has each of the 19 positions once, connected by legal moves. The search tree is what the algorithm builds from the start state, and the same position can appear in it many times.

**Frontier vs explored set:** the frontier holds nodes that are generated but not expanded yet. The explored set holds states that were already removed from the frontier and expanded.

**Tree search:** weakness is repeated states and cycles. Advantage is less memory and less bookkeeping, which is fine when the state space has no repeated states.

**Graph search:** cost is the extra memory for the explored set. Advantage is each state is expanded at most once, so it always finishes on a finite graph.